# Single-name spread moves for a CDX / iTraxx index

Pick one of the 40 indices in `cds_index.csv` and two dates. Output is one row per non-zero-weight constituent with `conv_spread` at date X, at date Y, and the change.

Needs a running Bloomberg Terminal (index roster + `MATURITY`) and a Databricks OAuth login (a browser window opens on the first pull of the day). Rosters and maturities are cached forever; the Databricks history is cached per day under `cache/`.

In [ ]:
import pandas as pd
import cdx_spreads as cs

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)
cs.load_index_list()

## Parameters

In [ ]:
INDEX = 'IG47'            # any index_name from the table above
DATE_X = '2026-09-30'     # start; snaps to the last available date on or before
DATE_Y = '2026-10-22'     # end;   same
METRIC = 'conv_spread'    # or par_spread, credit_dv01, risky_pv01, all_upfront

## Run

In [ ]:
res = cs.spread_changes(INDEX, DATE_X, DATE_Y, metric=METRIC)
a = res.attrs
print(f"{INDEX}: {a['metric']} on {a['date_x']:%Y-%m-%d} -> {a['date_y']:%Y-%m-%d}, index maturity {a['maturity']:%Y-%m-%d}")
print(f"{len(res)} names | median change {res['change'].median():.1f} | {res['change'].isna().sum()} without data")
res

## Checks
- `in_databricks == False`: the name is mapped but has no rows in `cds_transform_interpolated` at this maturity (Markit doesn't have it, or the ticker differs).
- `y_batch == 'EOD-Interp'`: the index maturity isn't a quoted Markit tenor on date Y, so the spread is interpolated between neighbouring tenors (common for older series).

In [ ]:
print('Not in Databricks:')
display(res.loc[~res['in_databricks'], ['member_name', 'Market_CDS_Ticker']])
print('Interpolated at date Y:', (res['y_batch'] == 'EOD-Interp').sum(), 'of', len(res))

## Largest moves

In [ ]:
res.dropna(subset=['change']).sort_values('change', key=abs, ascending=False).head(15)